# How LLMs Are Built — A Hands-On Companion

This notebook is the **run-it-yourself companion** to the *LLM Training Pipeline* slide deck. Every section below implements a small, honest, from-scratch version of a real concept — in plain NumPy, no frameworks, no GPU, no downloads — so you can actually watch the math happen instead of just reading about it.

**How to use this:** run the cells top to bottom (each section reuses helper functions defined earlier). Then go back and change the numbers — that's where the real learning happens. Try a different corpus for BPE, a different advantage sign for PPO, a different rank for LoRA, and see what changes.

Chapters mirror the deck:
1. Tokenization (BPE, Unigram/SentencePiece)
2. Pretraining (next-token prediction, cross-entropy)
3. Instruction Tuning (masked loss)
4. Alignment (RLHF reward model, PPO, DPO)
5. PEFT (LoRA, quantization)
6. Efficiency (distillation, Mixture of Experts)
7. Transformer Internals (embeddings, positions, attention, a full block)
8. Decoding & Prompting (sampling strategies, self-consistency)
9. Scaling & Systems (scaling laws, KV cache, gradient clipping, Adam, LR schedules)
10. Evaluation (perplexity, calibration)
11. Safety (watermarking)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter, defaultdict
import hashlib

# dark theme to match the deck
plt.rcParams['figure.facecolor'] = '#0a0a0f'
plt.rcParams['axes.facecolor']   = '#0a0a0f'
plt.rcParams['savefig.facecolor']= '#0a0a0f'
plt.rcParams['axes.edgecolor']   = '#444455'
plt.rcParams['text.color']       = '#dddde4'
plt.rcParams['axes.labelcolor']  = '#dddde4'
plt.rcParams['xtick.color']      = '#9999aa'
plt.rcParams['ytick.color']      = '#9999aa'
plt.rcParams['grid.color']       = '#222230'
plt.rcParams['axes.titlecolor']  = '#f0f0f5'

np.random.seed(42)
print("Ready.")

## 1 · Tokenization

### 1.1 Byte Pair Encoding (BPE) from scratch
BPE starts from characters and repeatedly merges the **most frequent adjacent pair** into a new symbol. Watch which pair wins at each step — it's always the most common one so far.

In [ ]:
corpus = {"low": 5, "lower": 2, "lowest": 4, "newer": 6, "wider": 3, "new": 2}

def word_to_symbols(word):
    return list(word) + ["</w>"]

vocab = {tuple(word_to_symbols(w)): freq for w, freq in corpus.items()}

def get_pair_counts(vocab):
    pairs = Counter()
    for symbols, freq in vocab.items():
        for i in range(len(symbols) - 1):
            pairs[(symbols[i], symbols[i + 1])] += freq
    return pairs

def merge_pair(pair, vocab):
    a, b = pair
    merged = a + b
    new_vocab = {}
    for symbols, freq in vocab.items():
        new_symbols, i = [], 0
        while i < len(symbols):
            if i < len(symbols) - 1 and symbols[i] == a and symbols[i + 1] == b:
                new_symbols.append(merged); i += 2
            else:
                new_symbols.append(symbols[i]); i += 1
        key = tuple(new_symbols)
        new_vocab[key] = new_vocab.get(key, 0) + freq
    return new_vocab

num_merges = 12
merges = []
print(f"{'step':<6}{'best pair':<18}{'count':<8}")
for step in range(num_merges):
    pairs = get_pair_counts(vocab)
    if not pairs:
        break
    best = max(pairs, key=pairs.get)
    print(f"{step:<6}{str(best):<18}{pairs[best]:<8}")
    merges.append(best)
    vocab = merge_pair(best, vocab)

print("\nLearned merges, in order:")
print(" | ".join("+".join(m) for m in merges))

print("\nFinal tokenized training words:")
for symbols, freq in vocab.items():
    print(f"  {' '.join(symbols):<28} (freq={freq})")

### 1.2 Using the learned merges on a brand-new word
The real test of BPE: apply the *same* learned merge rules to a word the tokenizer never saw during training (`"newest"`). It should still split sensibly, because it's built from pieces seen elsewhere.

In [ ]:
def bpe_encode(word, merges):
    symbols = word_to_symbols(word)
    for a, b in merges:
        merged = a + b
        new_symbols, i = [], 0
        while i < len(symbols):
            if i < len(symbols) - 1 and symbols[i] == a and symbols[i + 1] == b:
                new_symbols.append(merged); i += 2
            else:
                new_symbols.append(symbols[i]); i += 1
        symbols = new_symbols
    return symbols

for w in ["lowest", "newest", "wider", "wonderful"]:
    print(f"{w:<12} -> {bpe_encode(w, merges)}")

### 1.3 Unigram segmentation (the SentencePiece idea)
Instead of greedily merging, a Unigram tokenizer scores **every possible way to split a word** using per-piece probabilities, and keeps the split with the highest total probability. Below, we brute-force every valid segmentation of `"lower"` from a small toy vocabulary and rank them.

In [ ]:
word = "lower"
vocab_probs = {
    "l": 0.04, "o": 0.04, "w": 0.04, "e": 0.05, "r": 0.05,
    "lo": 0.05, "low": 0.20, "we": 0.03, "er": 0.15, "wer": 0.02, "ow": 0.02,
}

def all_segmentations(w):
    if w == "":
        yield []
        return
    for i in range(1, len(w) + 1):
        prefix, rest = w[:i], w[i:]
        if prefix in vocab_probs:
            for rest_seg in all_segmentations(rest):
                yield [prefix] + rest_seg

results = []
for seg in all_segmentations(word):
    p = np.prod([vocab_probs[piece] for piece in seg])
    results.append((seg, p))
results.sort(key=lambda x: -x[1])

best_seg = results[0][0]
print(f"All valid segmentations of '{word}':\n")
for seg, p in results:
    marker = "  <-- highest probability (chosen)" if seg == best_seg else ""
    print(f"  {' | '.join(seg):<22} P = {p:.6f}{marker}")

## 2 · Pretraining

### 2.1 Next-token prediction, at toy scale
Pretraining is "predict the next token" repeated at massive scale. Here's the smallest honest version: count how often each word follows each other word in a tiny corpus, turn that into a probability distribution, and sample from it — literally a language model, just a tiny one.

In [ ]:
text = "the cat sat on the mat the cat ran on the rug the dog sat on the mat the dog ran on the mat"
tokens = text.split()

bigram_counts = defaultdict(Counter)
for a, b in zip(tokens[:-1], tokens[1:]):
    bigram_counts[a][b] += 1

def next_token_distribution(word):
    counts = bigram_counts[word]
    total = sum(counts.values())
    return {w: c / total for w, c in counts.items()}

dist = next_token_distribution("the")
print("P(next word | 'the'):")
for w, p in sorted(dist.items(), key=lambda x: -x[1]):
    print(f"  {w:<8} {p:.2f}")

def sample_next(word, rng):
    dist = next_token_distribution(word)
    words, probs = zip(*dist.items())
    return rng.choice(words, p=probs)

rng = np.random.default_rng(0)
print("\nSampled continuations starting from 'the':")
for _ in range(4):
    w = "the"
    out = [w]
    for _ in range(7):
        if w not in bigram_counts:
            break
        w = sample_next(w, rng)
        out.append(w)
    print("  " + " ".join(out))

### 2.2 Cross-entropy loss: why confident-and-wrong hurts so much
The loss is just `-log(probability given to the correct token)`. Watch how sharply it punishes a *confidently wrong* prediction compared to a merely *unsure* one.

In [ ]:
probs = np.linspace(0.001, 1, 400)
loss = -np.log(probs)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(probs, np.clip(loss, 0, 6), color="#63b3ed", lw=2)

markers = [(0.9, "confident & correct", "#68d391"),
           (0.1, "unsure", "#e0a458"),
           (0.01, "confidently wrong", "#e57373")]
for p, label, color in markers:
    l = min(-np.log(p), 6)
    ax.scatter([p], [l], color=color, zorder=5)
    ax.annotate(f"{label}\np={p}, loss={-np.log(p):.2f}", (p, l),
                textcoords="offset points", xytext=(12, 8), color=color, fontsize=9)

ax.set_xlabel("probability assigned to the correct token")
ax.set_ylabel("loss = -log(p)")
ax.set_title("Cross-entropy loss vs. predicted probability")
plt.tight_layout(); plt.show()

## 3 · Instruction Tuning

### 3.1 Masked loss: only the response should teach the model
During SFT, the loss on **prompt** tokens is masked to zero — we only want the model to learn to write good *responses*, not to imitate how humans phrase *questions*.

In [ ]:
vocab_size, seq_len = 10, 8
rng = np.random.default_rng(1)
logits = rng.standard_normal((seq_len, vocab_size))
targets = rng.integers(0, vocab_size, seq_len)
is_response = np.array([0, 0, 0, 0, 1, 1, 1, 1], dtype=bool)  # first 4 = prompt, last 4 = response

def softmax(x):
    e = np.exp(x - x.max(axis=-1, keepdims=True))
    return e / e.sum(axis=-1, keepdims=True)

probs = softmax(logits)
token_losses = -np.log(probs[np.arange(seq_len), targets] + 1e-9)

print(f"{'position':<10}{'is_response':<14}{'loss':<8}")
for i in range(seq_len):
    print(f"{i:<10}{str(is_response[i]):<14}{token_losses[i]:.3f}")

print(f"\nFull-sequence average loss (WRONG for SFT): {token_losses.mean():.3f}")
print(f"Response-only masked loss   (CORRECT for SFT): {token_losses[is_response].mean():.3f}")

## 4 · Alignment: RLHF, PPO, DPO

### 4.1 The reward model: Bradley-Terry preference math
Given two responses and their reward scores, `sigmoid(r_A - r_B)` gives the probability that A is preferred. The loss pushes the winner's score up and the loser's down.

In [ ]:
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

r_A, r_B = 2.4, 0.3
p_a_wins = sigmoid(r_A - r_B)
loss = -np.log(p_a_wins)
print(f"r(A)={r_A}, r(B)={r_B}")
print(f"P(A preferred over B) = sigmoid(r_A - r_B) = {p_a_wins:.3f}")
print(f"Reward-model loss (A was the human-labeled winner): {loss:.3f}")

gaps = np.linspace(-4, 4, 200)
losses = -np.log(sigmoid(gaps))
plt.figure(figsize=(6, 3.5))
plt.plot(gaps, losses, color="#e0a458")
plt.scatter([r_A - r_B], [loss], color="#e0a458", zorder=5)
plt.xlabel("r(winner) - r(loser)"); plt.ylabel("loss")
plt.title("Reward-model loss vs. score gap")
plt.tight_layout(); plt.show()

### 4.2 PPO's clipped objective
PPO caps how much a single update can change the policy. Notice how the **clipped** curve (solid) flattens out once the ratio moves too far from 1, while the **unclipped** curve (dashed) keeps rewarding runaway ratios — that flattening is what keeps training stable.

In [ ]:
def ppo_objective(ratio, advantage, eps=0.2):
    unclipped = ratio * advantage
    clipped = np.clip(ratio, 1 - eps, 1 + eps) * advantage
    return np.minimum(unclipped, clipped)

ratios = np.linspace(0.4, 1.6, 200)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, adv, title in [(axes[0], 1.0, "Positive advantage (good response)"),
                        (axes[1], -1.0, "Negative advantage (bad response)")]:
    ax.plot(ratios, ratios * adv, '--', color='#888', label='unclipped')
    ax.plot(ratios, ppo_objective(ratios, adv), color='#e08a8a', lw=2, label='PPO clipped')
    ax.axvline(1.0, color='#444', lw=0.8)
    ax.set_title(title); ax.set_xlabel("ratio r(\u03b8)"); ax.legend()
plt.tight_layout(); plt.show()

### 4.3 DPO's loss, computed directly from log-probabilities
No reward model, no RL loop — just the log-probabilities the policy and the frozen reference model already assign to the winning (`w`) and losing (`l`) responses.

In [ ]:
def dpo_loss(logp_w_policy, logp_l_policy, logp_w_ref, logp_l_ref, beta=0.1):
    implicit_reward_w = logp_w_policy - logp_w_ref
    implicit_reward_l = logp_l_policy - logp_l_ref
    margin = beta * (implicit_reward_w - implicit_reward_l)
    return -np.log(sigmoid(margin)), margin

logp_w_policy, logp_l_policy = -1.2, -3.0   # policy already prefers the winner
logp_w_ref, logp_l_ref = -1.5, -1.8         # reference model was less decisive

loss, margin = dpo_loss(logp_w_policy, logp_l_policy, logp_w_ref, logp_l_ref)
print(f"implicit reward margin (scaled by beta): {margin:.3f}")
print(f"DPO loss: {loss:.3f}")

deltas = np.linspace(-2, 2, 200)
plt.figure(figsize=(6, 3.5))
plt.plot(deltas, -np.log(sigmoid(0.1 * deltas)), color="#63b3ed")
plt.scatter([margin / 0.1 * 0.1], [loss], color="#63b3ed", zorder=5)
plt.xlabel("(logp_w - logp_w_ref) - (logp_l - logp_l_ref)")
plt.ylabel("DPO loss")
plt.title("DPO loss falls as the policy favors the winning response more")
plt.tight_layout(); plt.show()

## 5 · Efficient Adaptation: LoRA & Quantization

### 5.1 LoRA: how much do you actually save?
Compare the parameter count of fully fine-tuning a `d x d` weight matrix against learning a rank-`r` LoRA update `B @ A` instead.

In [ ]:
d = 4096
ranks = [1, 2, 4, 8, 16, 32, 64]
full_params = d * d
print(f"Full fine-tune parameters for one {d}x{d} matrix: {full_params:,}\n")
print(f"{'rank r':<10}{'LoRA params (2dr)':<20}{'reduction factor':<18}")
for r in ranks:
    lora_params = 2 * d * r
    print(f"{r:<10}{lora_params:<20,}{full_params / lora_params:<18.1f}x")

# a concrete rank-8 update, stored as two thin matrices instead of one big one
rng = np.random.default_rng(2)
r = 8
B = rng.standard_normal((d, r)) * 0.01
A = rng.standard_normal((r, d)) * 0.01
delta_W = B @ A
print(f"\ndelta_W effective shape: {delta_W.shape}")
print(f"but only B{B.shape} and A{A.shape} are ever stored or trained")
print(f"parameters stored: {B.size + A.size:,}  instead of  {delta_W.size:,}")

### 5.2 Quantization: uniform bins vs. distribution-aware bins (the NF4 idea)
Real neural network weights cluster near zero (roughly Normal). Compare **uniform** 4-bit quantization against **quantile-based** 4-bit quantization (the idea behind NF4) on weights drawn from that kind of distribution — the quantile version should have noticeably lower error.

In [ ]:
def uniform_quantize(x, bits=4):
    levels = 2 ** bits - 1
    x_min, x_max = x.min(), x.max()
    scale = (x_max - x_min) / levels
    q = np.round((x - x_min) / scale)
    return q * scale + x_min

def quantile_quantize(x, bits=4):
    levels = 2 ** bits
    edges = np.percentile(x, np.linspace(0, 100, levels + 1))
    x_hat = np.zeros_like(x)
    for i in range(levels):
        lo, hi = edges[i], edges[i + 1]
        mask = (x >= lo) & (x <= hi) if i == levels - 1 else (x >= lo) & (x < hi)
        if mask.sum() > 0:
            x_hat[mask] = x[mask].mean()
    return x_hat

rng = np.random.default_rng(3)
weights = rng.standard_normal(200_000)  # realistic: most weights cluster near 0

uniform_hat = uniform_quantize(weights, bits=4)
quantile_hat = quantile_quantize(weights, bits=4)

mse_uniform = np.mean((weights - uniform_hat) ** 2)
mse_quantile = np.mean((weights - quantile_hat) ** 2)

print(f"Uniform 4-bit quantization MSE:  {mse_uniform:.5f}")
print(f"Quantile 4-bit quantization MSE: {mse_quantile:.5f}  (NF4-style)")
print(f"Quantile quantization is {mse_uniform / mse_quantile:.1f}x more accurate on this data")

## 6 · Efficiency: Distillation & Mixture of Experts

### 6.1 Temperature-scaled softmax (why distillation uses it)
Raising the temperature `T` flattens a probability distribution, revealing the "dark knowledge" about which wrong answers are *almost* right.

In [ ]:
def softmax_T(logits, T=1.0):
    z = logits / T
    e = np.exp(z - z.max())
    return e / e.sum()

logits = np.array([4.5, 2.0, 0.5, -1.0])
labels = ["cat", "dog", "fox", "car"]

print(f"{'T':<6}" + "".join(f"{l:<10}" for l in labels))
for T in [1, 2, 4, 8]:
    probs = softmax_T(logits, T)
    print(f"{T:<6}" + "".join(f"{p:<10.3f}" for p in probs))

fig, ax = plt.subplots(figsize=(6, 4))
x = np.arange(len(labels))
for T, color in zip([1, 2, 4, 8], ["#e57373", "#e0a458", "#63b3ed", "#68d391"]):
    ax.plot(x, softmax_T(logits, T), 'o-', label=f"T={T}", color=color)
ax.set_xticks(x); ax.set_xticklabels(labels)
ax.set_ylabel("probability"); ax.legend(); ax.set_title("Distributions flatten as temperature rises")
plt.tight_layout(); plt.show()

### 6.2 Mixture of Experts: top-k gating and load balancing
A gating network scores every expert; only the top-k highest scorers actually run. Then check: across many tokens, does routing stay balanced, or does it collapse onto a few favorite experts?

In [ ]:
expert_names = ["biology", "math", "code", "chit-chat", "history"]

def moe_route(gate_logits, top_k=2):
    probs = softmax_T(gate_logits, T=1.0)
    top_idx = np.argsort(-probs)[:top_k]
    top_probs = probs[top_idx]
    top_probs = top_probs / top_probs.sum()
    return top_idx, top_probs, probs

gate_logits = np.array([2.1, -0.5, -0.8, -1.0, 0.9])
idx, weights, all_probs = moe_route(gate_logits, top_k=2)

print("all gate probabilities:")
for n, p in zip(expert_names, all_probs):
    print(f"  {n:<12} {p:.3f}")
print("\nselected experts (top-2):")
for i, w in zip(idx, weights):
    print(f"  {expert_names[i]:<12} weight={w:.3f}")

# load balancing across a batch of 300 random tokens
rng = np.random.default_rng(4)
usage = np.zeros(5)
for _ in range(300):
    row = rng.standard_normal(5)
    sel, _, _ = moe_route(row, top_k=2)
    usage[sel] += 1
usage_frac = usage / usage.sum()

print("\nExpert usage across 300 tokens (top-2 routing, no load-balancing loss applied):")
for n, f in zip(expert_names, usage_frac):
    print(f"  {n:<12} {f:.1%}")

## 7 · Transformer Internals

### 7.1 Embeddings: geometry encodes meaning
We hand-craft embeddings so that related words share a "theme" vector plus noise, then check: does cosine similarity recover the categories (animals, vehicles, royalty) without ever being told what they are?

In [ ]:
rng = np.random.default_rng(5)
d = 8
themes = {"animal": rng.standard_normal(d), "vehicle": rng.standard_normal(d), "royalty": rng.standard_normal(d)}

emb = {
    "cat":   themes["animal"]  + rng.standard_normal(d) * 0.15,
    "dog":   themes["animal"]  + rng.standard_normal(d) * 0.15,
    "car":   themes["vehicle"] + rng.standard_normal(d) * 0.15,
    "truck": themes["vehicle"] + rng.standard_normal(d) * 0.15,
    "king":  themes["royalty"] + rng.standard_normal(d) * 0.15,
    "queen": themes["royalty"] + rng.standard_normal(d) * 0.15,
}
words = list(emb.keys())

def cos_sim(a, b):
    return a @ b / (np.linalg.norm(a) * np.linalg.norm(b))

sim_matrix = np.array([[cos_sim(emb[w1], emb[w2]) for w2 in words] for w1 in words])

fig, ax = plt.subplots(figsize=(5.5, 5))
im = ax.imshow(sim_matrix, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(words))); ax.set_xticklabels(words, rotation=45)
ax.set_yticks(range(len(words))); ax.set_yticklabels(words)
plt.colorbar(im, label="cosine similarity")
ax.set_title("Embedding similarity — clusters emerge from usage, not labels")
plt.tight_layout(); plt.show()

### 7.2 Sinusoidal positional encoding
Each position gets a unique pattern across the embedding dimensions — different frequencies of sine/cosine waves. This is what gets added to a token's embedding so the model can tell position 3 from position 30.

In [ ]:
def sinusoidal_pe(seq_len, d_model):
    pos = np.arange(seq_len)[:, None]
    i = np.arange(d_model)[None, :]
    angle_rates = 1 / (10000 ** (2 * (i // 2) / d_model))
    angles = pos * angle_rates
    pe = np.zeros((seq_len, d_model))
    pe[:, 0::2] = np.sin(angles[:, 0::2])
    pe[:, 1::2] = np.cos(angles[:, 1::2])
    return pe

pe = sinusoidal_pe(seq_len=50, d_model=64)

plt.figure(figsize=(7, 4))
plt.imshow(pe.T, aspect="auto", cmap="RdBu_r", vmin=-1, vmax=1)
plt.xlabel("position"); plt.ylabel("embedding dimension")
plt.title("Sinusoidal positional encoding")
plt.colorbar(label="value")
plt.tight_layout(); plt.show()

### 7.3 Self-attention from scratch
The full mechanism in eight lines: score every pair with `Q @ K.T`, scale, softmax each row, then blend `V` by those weights. We nudge `"it"`'s query to align with `"trophy"`'s key so the demo is legible — in a trained model, this alignment emerges from data, not a nudge.

In [ ]:
def softmax_rows(x):
    e = np.exp(x - x.max(axis=-1, keepdims=True))
    return e / e.sum(axis=-1, keepdims=True)

def self_attention(Q, K, V):
    d_k = Q.shape[-1]
    scores = Q @ K.T / np.sqrt(d_k)
    weights = softmax_rows(scores)
    return weights @ V, weights

rng = np.random.default_rng(6)
tokens = ["The", "trophy", "didn't", "fit", "because", "it"]
n, d_k = len(tokens), 8
Q = rng.standard_normal((n, d_k))
K = rng.standard_normal((n, d_k))
V = rng.standard_normal((n, d_k))
Q[5] = K[1] + rng.standard_normal(d_k) * 0.1   # make "it" look for "trophy"

output, weights = self_attention(Q, K, V)

plt.figure(figsize=(5.5, 5))
plt.imshow(weights, cmap="Blues")
plt.xticks(range(n), tokens, rotation=45); plt.yticks(range(n), tokens)
plt.colorbar(label="attention weight")
plt.title("Self-attention weights")
plt.tight_layout(); plt.show()

print(f"'it' attends most strongly to: '{tokens[np.argmax(weights[5])]}'")

### 7.4 Multi-head attention
Split Q/K/V into several smaller heads, run independent attention in each, then concatenate. Each head is free to specialize on a different kind of relationship.

In [ ]:
def multi_head_attention(X, num_heads, d_model, rng):
    d_head = d_model // num_heads
    Wq, Wk, Wv, Wo = (rng.standard_normal((d_model, d_model)) * 0.1 for _ in range(4))
    Q, K, V = X @ Wq, X @ Wk, X @ Wv
    n = X.shape[0]
    Qh = Q.reshape(n, num_heads, d_head).transpose(1, 0, 2)
    Kh = K.reshape(n, num_heads, d_head).transpose(1, 0, 2)
    Vh = V.reshape(n, num_heads, d_head).transpose(1, 0, 2)
    head_outputs = [self_attention(Qh[h], Kh[h], Vh[h])[0] for h in range(num_heads)]
    concat = np.concatenate(head_outputs, axis=-1)
    return concat @ Wo

rng = np.random.default_rng(7)
d_model, num_heads, n = 16, 4, 6
X = rng.standard_normal((n, d_model))
out = multi_head_attention(X, num_heads, d_model, rng)
print(f"input shape:  {X.shape}")
print(f"output shape: {out.shape}   (same shape — {num_heads} heads of size {d_model // num_heads} concatenated back to {d_model})")

### 7.5 A full transformer block, forward pass
Attention -> residual add -> LayerNorm -> feed-forward -> residual add -> LayerNorm. All untrained (random weights), but the *shapes* and the *normalization behavior* are exactly what a real block does.

In [ ]:
def layer_norm(x, eps=1e-5):
    mu = x.mean(axis=-1, keepdims=True)
    var = x.var(axis=-1, keepdims=True)
    return (x - mu) / np.sqrt(var + eps)

def relu(x):
    return np.maximum(0, x)

def transformer_block(X, num_heads, d_model, d_ff, rng):
    attn_out = multi_head_attention(X, num_heads, d_model, rng)
    X = layer_norm(X + attn_out)
    W1 = rng.standard_normal((d_model, d_ff)) * 0.1
    W2 = rng.standard_normal((d_ff, d_model)) * 0.1
    ffn_out = relu(X @ W1) @ W2
    return layer_norm(X + ffn_out)

rng = np.random.default_rng(8)
X = rng.standard_normal((6, 16))
out = transformer_block(X, num_heads=4, d_model=16, d_ff=64, rng=rng)

print(f"block input shape:  {X.shape}")
print(f"block output shape: {out.shape}")
print(f"\nper-token mean after LayerNorm (should be ~0): {out.mean(axis=-1).round(4)}")
print(f"per-token std  after LayerNorm (should be ~1): {out.std(axis=-1).round(4)}")

## 8 · Decoding & Prompting

### 8.1 Greedy vs. top-k vs. top-p sampling
Same probability distribution, four different ways to pick the next token. Watch how top-p's candidate pool size would change if the distribution were more or less "peaked".

In [ ]:
def top_k_filter(probs, k):
    idx = np.argsort(-probs)[:k]
    mask = np.zeros_like(probs, dtype=bool); mask[idx] = True
    filtered = np.where(mask, probs, 0)
    return filtered / filtered.sum()

def top_p_filter(probs, p):
    order = np.argsort(-probs)
    cum = np.cumsum(probs[order])
    cutoff = np.searchsorted(cum, p) + 1
    mask = np.zeros_like(probs, dtype=bool)
    mask[order[:cutoff]] = True
    filtered = np.where(mask, probs, 0)
    return filtered / filtered.sum()

words = ["the", "a", "this", "my", "that", "some"]
probs = np.array([0.45, 0.25, 0.12, 0.08, 0.06, 0.04])

print("distribution:      ", dict(zip(words, probs)))
print("greedy picks:       ", words[np.argmax(probs)])
print("top-k=3 renormalized:", dict(zip(words, top_k_filter(probs, 3).round(3))))
print("top-p=0.9 renormalized:", dict(zip(words, top_p_filter(probs, 0.9).round(3))))

rng = np.random.default_rng(9)
samples = [rng.choice(words, p=top_p_filter(probs, 0.9)) for _ in range(12)]
print("\n12 samples at top-p=0.9:", samples)

### 8.2 Self-consistency: does voting across samples actually help?
Simulate a "reasoner" that gets the right answer 60% of the time and a wrong one otherwise. Compare the accuracy of trusting **one** sample vs. taking a **majority vote** over `k` independent samples — run thousands of trials to get a reliable estimate.

In [ ]:
true_answer = 42
wrong_answers = [39, 44, 41]

def run_trial(k_samples, p_correct, n_trials=4000, seed=0):
    rng = np.random.default_rng(seed)
    single_correct = vote_correct = 0
    for _ in range(n_trials):
        samples = [true_answer if rng.random() < p_correct else rng.choice(wrong_answers)
                   for _ in range(k_samples)]
        if samples[0] == true_answer:
            single_correct += 1
        vals, counts = np.unique(samples, return_counts=True)
        if vals[np.argmax(counts)] == true_answer:
            vote_correct += 1
    return single_correct / n_trials, vote_correct / n_trials

print(f"{'k samples':<12}{'single-sample acc':<20}{'majority-vote acc':<20}")
for k in [1, 3, 5, 9, 15]:
    single_acc, vote_acc = run_trial(k, p_correct=0.6)
    print(f"{k:<12}{single_acc:<20.3f}{vote_acc:<20.3f}")

## 9 · Scaling & Systems

### 9.1 Fitting a scaling law
Generate synthetic loss numbers that follow a power law, then recover the exponent with a simple log-log linear fit — this is literally how researchers read scaling-law plots.

In [ ]:
rng = np.random.default_rng(10)
N = np.array([1e6, 1e7, 1e8, 1e9, 1e10, 1e11])
true_alpha, true_Nc = 0.34, 8.8e13
loss = (true_Nc / N) ** true_alpha * (1 + rng.standard_normal(len(N)) * 0.01)

log_N, log_loss = np.log(N), np.log(loss)
slope, intercept = np.polyfit(log_N, log_loss, 1)
alpha_fit = -slope
print(f"True alpha: {true_alpha}   Fitted alpha: {alpha_fit:.3f}")

plt.figure(figsize=(6, 4))
plt.plot(np.log10(N), log_loss, 'o', color="#e0a458", label="measured loss", ms=8)
plt.plot(np.log10(N), slope * log_N + intercept, '--', color="#888", label="power-law fit")
plt.xlabel("log10(model size)"); plt.ylabel("log(loss)")
plt.legend(); plt.title("Recovering a scaling-law exponent from data")
plt.tight_layout(); plt.show()

### 9.2 Why the KV cache matters — the actual cost curves
Without caching, generating token `L` means recomputing attention over all `L` previous tokens — the total work across a whole generation grows **quadratically**. With caching, it's **linear**.

In [ ]:
def naive_cost(seq_len):
    return sum(range(1, seq_len + 1))

def cached_cost(seq_len):
    return seq_len

lengths = np.arange(1, 200, 5)
naive = [naive_cost(l) for l in lengths]
cached = [cached_cost(l) for l in lengths]

plt.figure(figsize=(6, 4))
plt.plot(lengths, naive, color="#e08a8a", label="no KV cache  (O(L\u00b2))")
plt.plot(lengths, cached, color="#68d391", label="with KV cache (O(L))")
plt.xlabel("sequence length generated"); plt.ylabel("total attention 'work' units")
plt.legend(); plt.title("Cumulative generation cost, cached vs. not")
plt.tight_layout(); plt.show()

### 9.3 Gradient clipping in action
Eight gradient vectors, two of them artificially spiked. Clip every one to a max norm and watch only the spikes get rescaled.

In [ ]:
def clip_grad_norm(g, max_norm):
    norm = np.linalg.norm(g)
    return (g * (max_norm / norm) if norm > max_norm else g), norm

rng = np.random.default_rng(11)
grads = [rng.standard_normal(10) * 0.3 for _ in range(8)]
grads[3] *= 15   # outlier spikes
grads[6] *= 12

max_norm = 2.0
print(f"{'step':<6}{'raw norm':<12}{'clipped norm':<14}")
for i, g in enumerate(grads):
    clipped, raw_norm = clip_grad_norm(g, max_norm)
    print(f"{i:<6}{raw_norm:<12.2f}{np.linalg.norm(clipped):<14.2f}")

### 9.4 Adam vs. plain SGD on a hard loss surface
An elongated valley (steep in one direction, shallow in the other) is exactly where per-parameter adaptive step sizes shine. Watch SGD oscillate across the narrow axis while Adam glides down the valley.

In [ ]:
def loss_landscape(xy):
    x, y = xy
    return 0.05 * x**2 + 5 * y**2

def grad_fn(xy):
    x, y = xy
    return np.array([0.1 * x, 10 * y])

def run_sgd(start, lr=0.1, steps=40):
    path = [np.array(start, dtype=float)]
    for _ in range(steps):
        path.append(path[-1] - lr * grad_fn(path[-1]))
    return np.array(path)

def run_adam(start, lr=0.3, steps=40, b1=0.9, b2=0.999, eps=1e-8):
    x = np.array(start, dtype=float)
    m, v = np.zeros(2), np.zeros(2)
    path = [x.copy()]
    for t in range(1, steps + 1):
        g = grad_fn(x)
        m = b1 * m + (1 - b1) * g
        v = b2 * v + (1 - b2) * g**2
        m_hat, v_hat = m / (1 - b1**t), v / (1 - b2**t)
        x = x - lr * m_hat / (np.sqrt(v_hat) + eps)
        path.append(x.copy())
    return np.array(path)

start = [4.0, 1.0]
sgd_path, adam_path = run_sgd(start), run_adam(start)

xx, yy = np.meshgrid(np.linspace(-5, 5, 100), np.linspace(-1.5, 1.5, 100))
zz = 0.05 * xx**2 + 5 * yy**2

plt.figure(figsize=(7, 5))
plt.contour(xx, yy, zz, levels=20, colors='#333344', linewidths=0.6)
plt.plot(sgd_path[:, 0], sgd_path[:, 1], '-o', ms=3, color="#e08a8a", label="plain SGD")
plt.plot(adam_path[:, 0], adam_path[:, 1], '-o', ms=3, color="#68d391", label="Adam")
plt.scatter([0], [0], marker='*', s=200, color='white', zorder=5, label="minimum")
plt.legend(); plt.title("Optimizer trajectories on an elongated valley")
plt.tight_layout(); plt.show()

print(f"SGD  final distance to minimum:  {np.linalg.norm(sgd_path[-1]):.3f}")
print(f"Adam final distance to minimum:  {np.linalg.norm(adam_path[-1]):.3f}")

### 9.5 Learning rate schedule: warmup + cosine decay
The same shape used to train nearly every large model.

In [ ]:
def lr_schedule(step, total_steps, warmup_frac=0.1, lr_max=1.0, lr_min=0.0):
    warmup_steps = int(total_steps * warmup_frac)
    if step < warmup_steps:
        return lr_max * step / warmup_steps
    progress = (step - warmup_steps) / (total_steps - warmup_steps)
    return lr_min + 0.5 * (lr_max - lr_min) * (1 + np.cos(np.pi * progress))

steps = np.arange(0, 1000)
lrs = [lr_schedule(s, 1000) for s in steps]

plt.figure(figsize=(6, 3.5))
plt.plot(steps, lrs, color="#63b3ed")
plt.xlabel("training step"); plt.ylabel("learning rate")
plt.title("Warmup + cosine decay schedule")
plt.tight_layout(); plt.show()

## 10 · Evaluation

### 10.1 Perplexity: turning loss into an interpretable number
Perplexity is just `exp(average cross-entropy loss)`. Compare a "good" and a "bad" toy model's token-level probabilities on the same held-out sequence.

In [ ]:
def perplexity(token_log_probs):
    return np.exp(-np.mean(token_log_probs))

good_model_probs = np.array([0.80, 0.75, 0.90, 0.60, 0.85])
bad_model_probs  = np.array([0.20, 0.15, 0.30, 0.10, 0.25])

print(f"Good model perplexity: {perplexity(np.log(good_model_probs)):.2f}")
print(f"Bad model perplexity:  {perplexity(np.log(bad_model_probs)):.2f}")
print("\n(lower perplexity = the model was less 'surprised', on average, by the real next token)")

### 10.2 Calibration: a reliability diagram
Simulate a model whose *stated* confidence is systematically higher than its *actual* accuracy — classic overconfidence — and plot it against the diagonal "perfectly calibrated" line.

In [ ]:
rng = np.random.default_rng(12)
n = 3000
confidences = rng.beta(2, 2, n)
true_correct_prob = confidences * 0.7          # model is overconfident by a factor of 0.7
outcomes = (rng.random(n) < true_correct_prob).astype(int)

bins = np.linspace(0, 1, 11)
bin_idx = np.digitize(confidences, bins) - 1
bin_conf, bin_acc = [], []
for b in range(10):
    mask = bin_idx == b
    if mask.sum() > 0:
        bin_conf.append(confidences[mask].mean())
        bin_acc.append(outcomes[mask].mean())

plt.figure(figsize=(5.5, 5.5))
plt.plot([0, 1], [0, 1], '--', color='#666', label='perfect calibration')
plt.plot(bin_conf, bin_acc, 'o-', color="#e08a8a", label='this model')
plt.xlabel('stated confidence'); plt.ylabel('actual accuracy')
plt.legend(); plt.title('Reliability diagram — this model is overconfident')
plt.tight_layout(); plt.show()

ece = np.mean(np.abs(np.array(bin_conf) - np.array(bin_acc)))
print(f"Expected Calibration Error (ECE): {ece:.3f}")

## 11 · Safety: Watermarking Generated Text

### 11.1 Green-list / red-list watermarking, end to end
Before each token, split the vocabulary into a "green list" seeded by the *previous* token, then bias generation toward it. A detector that knows the same seeding rule can measure how much a piece of text skews green — and tell watermarked text apart from random ("human") text using a z-score.

In [ ]:
vocab = [f"tok{i}" for i in range(20)]

def green_list_for(prev_token, green_fraction=0.5, seed_salt=0):
    h = int(hashlib.sha256(f"{prev_token}-{seed_salt}".encode()).hexdigest(), 16)
    rng = np.random.default_rng(h % (2**32))
    n_green = int(len(vocab) * green_fraction)
    return set(rng.choice(len(vocab), size=n_green, replace=False))

def generate_watermarked(length, bias=4.0, seed=0):
    rng = np.random.default_rng(seed)
    tokens = [int(rng.integers(0, len(vocab)))]
    for _ in range(length - 1):
        green = green_list_for(tokens[-1])
        base_logits = rng.normal(0, 1, len(vocab))
        boosted = np.array([base_logits[i] + (bias if i in green else 0) for i in range(len(vocab))])
        p = np.exp(boosted - boosted.max()); p /= p.sum()
        tokens.append(int(rng.choice(len(vocab), p=p)))
    return tokens

def detect_watermark(tokens, green_fraction=0.5):
    green_hits = 0
    for i in range(1, len(tokens)):
        if tokens[i] in green_list_for(tokens[i - 1]):
            green_hits += 1
    n = len(tokens) - 1
    expected = n * green_fraction
    std = np.sqrt(n * green_fraction * (1 - green_fraction))
    z = (green_hits - expected) / std
    return green_hits, n, z

wm_tokens = generate_watermarked(300, bias=4.0, seed=0)
hits, n, z = detect_watermark(wm_tokens)
print(f"Watermarked text:  {hits}/{n} tokens on the green list  ->  z = {z:.2f}")

rng = np.random.default_rng(99)
random_tokens = list(rng.integers(0, len(vocab), 300))
hits2, n2, z2 = detect_watermark(random_tokens)
print(f"Random ('human') text: {hits2}/{n2} tokens on the green list  ->  z = {z2:.2f}")
print("\n(z well above ~4 is a strong watermark signal; z near 0 looks like ordinary text)")

---
## Where to go from here

Every demo above used toy data and untrained weights on purpose — the goal was to make the *mechanism* visible, not to train a real model. Natural next steps:

- Swap in a real corpus for the BPE section and watch the merges change.
- Replace the random transformer-block weights with weights loaded from a small real model (e.g. via `transformers` + `torch`, if you install them) and re-run the attention visualization on real text.
- Increase `k` in the self-consistency demo and see where the accuracy gain saturates.
- Try a *steeper* valley in the Adam vs. SGD demo and watch the gap widen further.

Go back to the deck (`llm_training_pipeline.html`) for the visual walkthrough of anything that needs a refresher before you tweak the code here.